# NirmaanAI — Improved Infrastructure Delay Regression Model
**Dataset**: `NirmaanAI_Feature_Engineered.csv` (182 projects total; 134 historical labeled, 48 ongoing unlabeled)
**Target Variable**: `Schedule Delay Months`
**ML Boundary**: 10 Historical-Safe Features (Strictly zero post-outcome temporal leakage)

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load Dataset
df = pd.read_csv('NirmaanAI_Feature_Engineered.csv')
print('Dataset loaded:', df.shape)

# Filter labeled projects (134 rows with known schedule delay)
target_col = [c for c in df.columns if 'Schedule Delay Months' in c.replace('\n', ' ') or 'Schedule Delay Months' in c][0]
labeled_df = df[df[target_col].notna()].copy()
print('Labeled projects count:', len(labeled_df))
print('Unlabeled/Ongoing projects count:', len(df) - len(labeled_df))

In [2]:
approved_features = ['Original Cost\n(in cr.)', 'Expenditure\n(in cr.)', 'Physical Progress\n(in %)', 'Expenditure %', 'Remaining Budget\n(in cr.)', 'Progress Gap', 'Planned Duration Days', 'Sector Name', 'Line Ministry', 'Implementing Agency']
numeric_features = approved_features[:7]
categorical_features = approved_features[7:]

X = labeled_df[approved_features].copy()
y = labeled_df[target_col].copy()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Features prepared successfully. Features count:', len(approved_features))

In [3]:
candidate_models = {
    'Baseline Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'Tuned Random Forest': RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_split=4, min_samples_leaf=2, max_features=0.7, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=300, max_depth=12, min_samples_split=4, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=6, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in candidate_models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    cv_res = cross_validate(pipe, X, y, cv=kf, scoring={'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error', 'r2': 'r2'}, n_jobs=-1)
    results.append({
        'Model': name,
        'Mean MAE': -cv_res['test_mae'].mean(),
        'MAE Std': cv_res['test_mae'].std(),
        'Mean RMSE': -cv_res['test_rmse'].mean(),
        'Mean R²': cv_res['test_r2'].mean()
    })

res_df = pd.DataFrame(results)
display(res_df.round(4))

In [4]:
# Train Final Selected Model on all 134 labeled projects
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', candidate_models['Extra Trees'])
])
final_model.fit(X, y)

# DHUPTALA OC Prediction
dhuptala_mask = df['Project Name'].str.contains('DHUPTALA', case=False, na=False) | (df['Project Code'] == 400353)
dhuptala_row = df[dhuptala_mask].iloc[0]
dhuptala_df = pd.DataFrame([dhuptala_row[approved_features]])
pred = final_model.predict(dhuptala_df)[0]

print('=== DHUPTALA OC TEST PREDICTION ===')
print(f'Project Code: {dhuptala_row["Project Code"]}')
print(f'Predicted Delay: {pred:.2f} months')

# Save improved model and metadata
joblib.dump(final_model, 'NirmaanAI_Final_RF_Regression_Improved.pkl')
joblib.dump({
    'features': approved_features,
    'model_name': 'Extra Trees'
}, 'NirmaanAI_ML_Features_Improved.pkl')
print('Artifacts exported successfully!')